# RAG Demo: Retrieve, Augment, Generate

This notebook implements a tiny dependency-free RAG pipeline. It retrieves relevant documents with token overlap, builds a context window, and creates an extractive answer from that context.

In [ ]:
import re
from collections.abc import Iterable


def tokenize(text: str) -> set[str]:
    return set(re.findall(r"[a-z0-9]+", text.lower()))


print("RAG demo initialized")

: 

In [ ]:
documents = [
    {
        "id": "doc-1",
        "title": "Factorio blueprints",
        "text": "Factorio blueprints describe factory layouts and can be shared with other players.",
    },
    {
        "id": "doc-2",
        "title": "Retrieval augmented generation",
        "text": "RAG retrieves relevant context before generating an answer, which helps ground responses in source material.",
    },
    {
        "id": "doc-3",
        "title": "Python notebooks",
        "text": "Python notebooks are useful for interactive experiments because code and results stay together.",
    },
]

for document in documents:
    document["tokens"] = tokenize(document["text"])

documents

In [ ]:
def retrieve(query: str, documents: Iterable[dict], limit: int = 2) -> list[dict]:
    query_tokens = tokenize(query)
    ranked = []

    for document in documents:
        overlap = query_tokens & document["tokens"]
        if overlap:
            ranked.append((len(overlap), document))

    ranked.sort(key=lambda item: item[0], reverse=True)
    return [document for _, document in ranked[:limit]]


def answer(query: str, documents: Iterable[dict]) -> str:
    retrieved = retrieve(query, documents)
    if not retrieved:
        return "I could not find supporting context."

    context = " ".join(document["text"] for document in retrieved)
    query_tokens = tokenize(query)
    sentences = re.split(r"(?<=[.!?])\s+", context)
    best_sentence = max(sentences, key=lambda sentence: len(query_tokens & tokenize(sentence)))

    print("Retrieved context:")
    for document in retrieved:
        print(f"- [{document['id']}] {document['title']}: {document['text']}")
    return f"Answer: {best_sentence}"


query = "How does RAG improve answers?"
print(answer(query, documents))